In [1]:
# env + data staging for UNETR (Model B) training
import os, glob, json, subprocess
try:
    import monai
    print('monai', monai.__version__)
except Exception:
    import sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'monai>=1.5'], check=True)
    import monai
    print('monai installed', monai.__version__)
import torch
print('torch', torch.__version__, '| cuda:', torch.cuda.is_available(), '| gpus:', torch.cuda.device_count())

os.chdir('/kaggle/working')
if not os.path.isdir('Neuroshield'):
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/pankurajasaivardhan/Neuroshield.git'], check=True)

# incremental history dump so a session cut still leaves history + best checkpoint
SRC = '/kaggle/working/Neuroshield/track2_brain_tumor/src/train_model_b_unetr.py'
s = open(SRC).read()
old = "history.append({'epoch': epoch + 1, 'train_loss': epoch_loss, 'val_dice': md})"
assert old in s
s = s.replace(old, old + "\n            json.dump({'best_val_dice': best_dice, 'history': history}, open(os.path.join(args.out, 'unetr_history.json'), 'w'), indent=2)")
open(SRC, 'w').write(s)

DS = '/kaggle/input/datasets/rajasaivardhan/neuroshield-track2-model-a'
splits = json.load(open(sorted(glob.glob(os.path.join(DS, '**/nnUNet_preprocessed/**/splits_final.json'), recursive=True) or glob.glob(os.path.join(DS, '**/splits_final.json'), recursive=True))[0]))
fold = splits[0]
test_ids = sorted({os.path.basename(p).replace('.nii.gz', '').replace('.nii', '')
                   for p in glob.glob(os.path.join(DS, '**/test_predictions/**/*.nii*'), recursive=True)})
print('fold0 train/val:', len(fold['train']), len(fold['val']), '| test:', len(test_ids))
assert len(fold['train']) == 249 and len(fold['val']) == 63 and len(test_ids) == 56, 'split mismatch vs training_metadata'
rows = [(c, 'train') for c in fold['train']] + [(c, 'val') for c in fold['val']] + [(c, 'test') for c in test_ids]
import pandas as pd
pd.DataFrame(rows, columns=['case_id', 'split']).to_csv('/kaggle/working/manifest.csv', index=False)

GT = glob.glob('/kaggle/input/datasets/awsaf49/brats20-dataset-training-validation/**/MICCAI_BraTS2020_TrainingData', recursive=True)[0]
stage = '/kaggle/working/brats'
MODS = ['flair', 't1', 't1ce', 't2']
n = 0
for cid, _ in rows:
    cdir = os.path.join(GT, cid)
    if not os.path.isdir(cdir):
        print('MISSING', cid)
        continue
    out = os.path.join(stage, cid)
    os.makedirs(out, exist_ok=True)
    for m in MODS + ['seg']:
        src = os.path.join(cdir, f'{cid}_{m}.nii')
        dst = os.path.join(out, f'{cid}_{m}.nii')
        if not os.path.exists(dst):
            os.symlink(os.path.realpath(src), dst)
    n += 1
print('staged cases:', n)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 52.3 MB/s eta 0:00:00
monai installed 1.6.1
torch 2.11.0+cu128 | cuda: True | gpus: 2


Cloning into 'Neuroshield'...


fold0 train/val: 249 63 | test: 56
staged cases: 368


In [2]:
# train UNETR (40 epochs, val every 10) on the Model A 249/63 split (test 56)
import subprocess, os, json
os.chdir('/kaggle/working/Neuroshield')
r = subprocess.run(['python', '-m', 'track2_brain_tumor.src.train_model_b_unetr',
                    '--manifest', '/kaggle/working/manifest.csv',
                    '--raw-data', '/kaggle/working/brats',
                    '--out', '/kaggle/working/outputs',
                    '--epochs', '40', '--val-interval', '10'])
print('exit', r.returncode)
assert r.returncode == 0, 'UNETR training failed - see log'
p = '/kaggle/working/outputs/unetr_history.json'
if os.path.exists(p):
    h = json.load(open(p))
    print('best_val_dice:', h['best_val_dice'], '| epochs logged:', len(h['history']))

nvImageCodec is not available; skipping pydicom decoder plugin registration.
/usr/local/lib/python3.13/dist-packages/monai/data/image_reader.py:1166: UserWarning: NvImgCodecPydicomReader: nvImageCodec decoder plugin did not register successfully. Falling back to default pydicom decoders.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/monai/utils/deprecate_utils.py:320: FutureWarning: monai.transforms.spatial.dictionary Orientationd.__init__:labels: Current default value of argument `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` was changed in version None from `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` to `labels=None`. Default value changed to None meaning that the transform now uses the 'space' of a meta-tensor, if applicable, to determine appropriate axis labels.
  warn_deprecated(argname, msg, warning_category)
nvImageCodec is not available; skipping pydicom decoder plugin registration.
/usr/local/lib/python3.13/dist-packages/monai/data/image_reader.py:1166: UserWarning

epoch 10: loss=0.6865 val_dice=0.5765
  new best val Dice 0.5765, checkpoint saved
epoch 20: loss=0.5375 val_dice=0.6069
  new best val Dice 0.6069, checkpoint saved
epoch 30: loss=0.4830 val_dice=0.6159
  new best val Dice 0.6159, checkpoint saved
epoch 40: loss=0.4470 val_dice=0.6305
  new best val Dice 0.6305, checkpoint saved
Done. Best val Dice 0.6305. Next: held-out test eval with eval_model_a_test.py adapted for UNETR, then reliability comparison.
exit 0
best_val_dice: 0.6305475831031799 | epochs logged: 4
